In [ ]:
import os
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans
from joblib import Parallel, delayed

# Make both the repository root and the Diffusion module discoverable when the
# notebook is executed from different working directories.
cwd = pathlib.Path().resolve()
repo_root = cwd if (cwd / 'XYModel').exists() else cwd.parent
for path in [repo_root, repo_root / 'Diffusion']:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from diffusion_map import diffusion_map, compute_eigendecomposition, construct_transition_matrix, compute_gaussian_kernel, construct_diffusion_map_embedding
from XYModel.xy import XYModelMetropolisSimulation

try:
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    use_gpu = torch.cuda.is_available()
    print(f"Using torch device: {device}")
except ImportError:
    torch = None
    device = 'cpu'
    use_gpu = False
    print('PyTorch not installed; falling back to CPU.')

print("=" * 80)
print("STEP 1: LOAD AND TRANSFORM DATA")
print("=" * 80)

               
configs = np.load('xy_model_configurations.npy')
metadata = pd.read_csv('xy_model_metadata.csv')

print(f"\nLoaded configurations: {configs.shape}")
print(f"Loaded metadata: {metadata.shape}")

                                                           
                                                        
print("\nTransforming spin angles to (cos θ, sin θ) representation...")
configs_transformed = np.zeros((configs.shape[0], configs.shape[1] * 2))

for i in range(configs.shape[0]):
    theta = configs[i, :]
    configs_transformed[i, ::2] = np.cos(theta)                        
    configs_transformed[i, 1::2] = np.sin(theta)                      

print(f"Transformed data shape: {configs_transformed.shape}")
print(f"  Original: {configs.shape[0]} configs × {configs.shape[1]} angles")
print(f"  Transformed: {configs_transformed.shape[0]} configs × {configs_transformed.shape[1]} coordinates")
print(f"  (Each angle θ → (cos θ, sin θ))")

                       
sample_idx = 0
print(f"\nVerification - first 3 spins of configuration {sample_idx}:")
for j in range(3):
    theta = configs[sample_idx, j]
    cos_theta = configs_transformed[sample_idx, 2*j]
    sin_theta = configs_transformed[sample_idx, 2*j+1]
    print(f"  Spin {j}: θ = {theta:.4f} → (cos θ, sin θ) = ({cos_theta:.4f}, {sin_theta:.4f})")
    print(f"           cos²θ + sin²θ = {cos_theta**2 + sin_theta**2:.6f} (should be 1.0)")


print("=" * 80)
print("STEP 2: REPRODUCE FIG 2 - EIGENVALUE SPECTRUM vs EPSILON")
print("=" * 80)

# Focus on the lowest temperature slice for phase-sensitive diagnostics
low_temp_mask = abs(metadata['temperature'] - 0.81) <= 1e-2
X_low_temp = configs_transformed[low_temp_mask]
metadata_low_temp = metadata[low_temp_mask]

print(f"Low temperature (T/J = 0.5) data:")
print(f"  Number of configurations: {X_low_temp.shape[0]}")
print(f"  Feature dimension: {X_low_temp.shape[1]}")
print(f"  Topological sectors:")
for sector in metadata_low_temp['sector_label'].unique():
    count = (metadata_low_temp['sector_label'] == sector).sum()
    print(f"    {sector}: {count} samples")



Using torch device: cuda
STEP 1: LOAD AND TRANSFORM DATA

Loaded configurations: (50000, 1024)
Loaded metadata: (50000, 6)

Transforming spin angles to (cos θ, sin θ) representation...
Transformed data shape: (50000, 2048)
  Original: 50000 configs × 1024 angles
  Transformed: 50000 configs × 2048 coordinates
  (Each angle θ → (cos θ, sin θ))

Verification - first 3 spins of configuration 0:
  Spin 0: θ = 0.1233 → (cos θ, sin θ) = (0.9924, 0.1230)
           cos²θ + sin²θ = 1.000000 (should be 1.0)
  Spin 1: θ = 0.4913 → (cos θ, sin θ) = (0.8817, 0.4718)
           cos²θ + sin²θ = 1.000000 (should be 1.0)
  Spin 2: θ = 6.2224 → (cos θ, sin θ) = (0.9982, -0.0608)
           cos²θ + sin²θ = 1.000000 (should be 1.0)
STEP 2: REPRODUCE FIG 2 - EIGENVALUE SPECTRUM vs EPSILON
Low temperature (T/J = 0.5) data:
  Number of configurations: 1000
  Feature dimension: 2048
  Topological sectors:
    (0,0): 200 samples
    (1,0): 200 samples
    (0,1): 200 samples
    (-1,0): 200 samples
    (-1,-1)

In [ ]:
from typing import List, Sequence, Tuple, Optional, Dict
from dataclasses import dataclass, asdict

import warnings
warnings.filterwarnings("ignore")

def adjusted_mse(K: np.ndarray, labels: np.ndarray) -> float:
    """Adjusted MSE between similarity matrix and an ideal block-diagonal target."""
    labels = np.asarray(labels)
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels)

    if n_clusters < 2:
        return np.inf  # Cannot compute adjusted MSE with fewer than 2 clusters

    ideal = (labels[:, None] == labels[None, :]).astype(float)
    diff_sq = (K - ideal) ** 2

    mse = 0.0
    for lbl in unique_labels:
        cluster_error = 0.0
        for i in labels: 
            if  i != lbl:
                continue
            for j in range(len(diff_sq[i])): 
                cluster_error += diff_sq[i][j]
        mse += cluster_error / len(labels[labels == lbl])

        # mask = labels == lbl
        # # Weight each cluster equally regardless of size.
        # mse += diff_sq[mask].sum() / mask.sum()

        # print(diff_sq[mask, :])
        # print(np.max(diff_sq[mask, :]))
    return (n_clusters - 1) / n_clusters * mse

def compute_gaussian_kernel(
    X: np.ndarray,
    epsilon: float,
    N: int = 1024,
    use_torch: bool = False,
    device: Optional[str] = None,
    return_torch: bool = False,
) -> np.ndarray:
    """
    Compute the Gaussian kernel matrix K from input samples.

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
        Input data matrix where each row is a sample (spin configuration)
    epsilon : float
        Bandwidth hyperparameter for the Gaussian kernel
    N : int, default=1024
        Number of spins (used for normalization). Default is 1024 for 32x32 lattice

    Returns
    -------
    K : np.ndarray, shape (n_samples, n_samples)
        Gaussian kernel matrix

    Notes
    -----
    The kernel is computed as: K_ij = exp(-||x_i - x_j||^2 / (2 * N * epsilon))
    Following the paper's specification for the XY model diffusion map analysis.
    """
    if use_torch:
        if torch is None:  # pragma: no cover
            raise ImportError("PyTorch is not installed; cannot use GPU backend.")
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        X_t = torch.as_tensor(X, device=device, dtype=torch.float64)
        pairwise_sq_dists = torch.cdist(X_t, X_t, p=2) ** 2
        K_t = torch.exp(-pairwise_sq_dists / (epsilon))
        if return_torch:
            return K_t
        return K_t.cpu().numpy()

    pairwise_sq_dists = squareform(pdist(X, metric='sqeuclidean'))
    K = np.exp(-pairwise_sq_dists / (epsilon))
    return K

def construct_transition_matrix(K):
    """
    Construct the row-stochastic transition matrix P from the kernel matrix K.

    Parameters
    ----------
    K : np.ndarray, shape (n_samples, n_samples)
        Gaussian kernel matrix

    Returns
    -------
    P : np.ndarray, shape (n_samples, n_samples)
        Row-stochastic transition matrix (each row sums to 1)

    Notes
    -----
    The transition matrix is constructed by normalizing each row of K:
    P_ij = K_ij / sum_j(K_ij)
    """
    # if torch is not None and isinstance(K, torch.Tensor):
    #     row_sums = K.sum(dim=1, keepdim=True)
    #     if torch.any(row_sums == 0):
    #         warnings.warn("Warning: Some rows of kernel matrix sum to zero.")
    #         row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
    #     P = K / row_sums
    #     return P

    row_sums = K.sum(axis=1, keepdims=True)
    if np.any(row_sums == 0):
        warnings.warn("Warning: Some rows of kernel matrix sum to zero.")
        row_sums[row_sums == 0] = 1.0
    P = K / row_sums

    return P


def compute_eigendecomposition(P, n_components: int = 10,
                                use_sparse: bool = False,
                                use_torch: bool = False,
                                device: Optional[str] = None,
                                return_torch: bool = False) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute eigenvalues and right eigenvectors of the transition matrix P.

    Parameters
    ----------
    P : np.ndarray, shape (n_samples, n_samples)
        Row-stochastic transition matrix
    n_components : int, default=10
        Number of leading eigenvectors to compute
    use_sparse : bool, default=False
        If True, use sparse eigendecomposition (efficient for large matrices)

    Returns
    -------
    eigenvalues : np.ndarray, shape (n_components,) or (n_samples,)
        Eigenvalues sorted in descending order by magnitude
    eigenvectors : np.ndarray, shape (n_samples, n_components) or (n_samples, n_samples)
        Right eigenvectors corresponding to the eigenvalues (column vectors)
    """
    n_samples = P.shape[0]

    if use_torch or (torch is not None and isinstance(P, torch.Tensor)):
        if torch is None:  # pragma: no cover
            raise ImportError("PyTorch is not installed; cannot use GPU backend.")
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        P_t = P if isinstance(P, torch.Tensor) else torch.as_tensor(P, device=device, dtype=torch.float64)
        if use_sparse:
            raise ValueError("Sparse eigendecomposition is not supported with the torch backend.")

        eigenvalues, eigenvectors = torch.linalg.eig(P_t.T)
        idx = torch.argsort(torch.abs(eigenvalues), descending=True)
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        if n_components < n_samples:
            eigenvalues = eigenvalues[:n_components]
            eigenvectors = eigenvectors[:, :n_components]

        if not return_torch:
            eigenvalues_np = eigenvalues.detach().cpu().numpy()
            eigenvectors_np = eigenvectors.detach().cpu().numpy()
            if np.allclose(eigenvalues_np.imag, 0):
                eigenvalues_np = eigenvalues_np.real
            if np.allclose(eigenvectors_np.imag, 0):
                eigenvectors_np = eigenvectors_np.real
            return eigenvalues_np, eigenvectors_np

        # For torch return type, keep complex dtype; caller can convert if desired.
        return eigenvalues, eigenvectors

    if use_sparse and n_components < n_samples:
        eigenvalues, eigenvectors = eigs(P.T, k=n_components, which='LM')
        idx = np.argsort(np.abs(eigenvalues))[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
    else:
        eigenvalues, eigenvectors = np.linalg.eig(P.T)
        idx = np.argsort(np.abs(eigenvalues))[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        if n_components < n_samples:
            eigenvalues = eigenvalues[:n_components]
            eigenvectors = eigenvectors[:, :n_components]

    if np.allclose(eigenvalues.imag, 0):
        eigenvalues = eigenvalues.real
    if np.allclose(eigenvectors.imag, 0):
        eigenvectors = eigenvectors.real

    return eigenvalues, eigenvectors

@dataclass
class TuningResult:
    epsilon: float
    n_clusters: int
    mse: float
    epsilon_idx: int
    cluster_idx: int
    eigenvalues: np.ndarray
    labels: np.ndarray

def scan_resolution_and_clusters(
    X: np.ndarray,
    epsilon_values: Sequence[float],
    cluster_options: Sequence[int],
    t: int = 1,
    n_components: int = 15,
    random_state: Optional[int] = 0,
    n_jobs: Optional[int] = None,
    use_torch: bool = False,
    device: Optional[str] = None,
) -> Tuple[TuningResult, List[TuningResult], np.ndarray]:
    """Evaluate adjusted MSE over a grid of epsilon and k-means cluster counts."""
    results: List[TuningResult] = []
    best: Optional[TuningResult] = None
    mse_surface = np.full((len(cluster_options), len(epsilon_values)), np.nan)

    def _evaluate_single_epsilon(eps_idx: int, eps: float) -> Tuple[int, List[TuningResult]]:
        K = compute_gaussian_kernel(
            X,
            epsilon=eps,
            N=X.shape[1],
            use_torch=use_torch,
            device=device,
            return_torch=False,
        )
        P = construct_transition_matrix(K)
        evals, evecs = compute_eigendecomposition(
            P,
            n_components=min(n_components, X.shape[0] - 1),
            use_torch=use_torch,
            device=device,
            return_torch=False,
        )
        n_dims = min(3, len(evals) - 1)
        embedding = construct_diffusion_map_embedding(
            evals, evecs, n_dimensions=n_dims, t=t, skip_first=True
        )
        embedding = np.real_if_close(embedding)

        eps_results: List[TuningResult] = []
        for n_idx, n_clusters in enumerate(cluster_options):
            kmeans = KMeans(n_clusters=n_clusters, n_init=10000, random_state=random_state)
            labels = kmeans.fit_predict(embedding)
            mse_val = adjusted_mse(K, labels)
            eps_results.append(
                TuningResult(
                    epsilon=eps,
                    n_clusters=n_clusters,
                    mse=mse_val,
                    epsilon_idx=eps_idx,
                    cluster_idx=n_idx,
                    eigenvalues=evals,
                    labels=labels,
                )
            )
        return eps_idx, eps_results

    if n_jobs is None or n_jobs == 1:
        eps_outputs = [_evaluate_single_epsilon(idx, eps) for idx, eps in enumerate(epsilon_values)]
    else:
        eps_outputs = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(_evaluate_single_epsilon)(idx, eps) for idx, eps in enumerate(epsilon_values)
        )

    for eps_idx, eps_results in eps_outputs:
        for result in eps_results:
            results.append(result)
            mse_surface[result.cluster_idx, eps_idx] = result.mse
            if best is None or result.mse < best.mse:
                best = result

    if best is None:
        raise RuntimeError("Tuning scan produced no results.")
    return best, results, mse_surface


n_clusters = 2
n_components = 1000000
t = 500000000
# t = 1
eps = 8
use_torch = use_gpu and (torch is not None)

X = X_low_temp
best_result, grid_results, mse_surface = scan_resolution_and_clusters(
    X,
    np.arange(0.1, 10.0, 0.1),
    [2, 3, 4, 5, 6, 7],
    t=t,
    n_components=15,
    random_state=0,
    n_jobs=-1,
    use_torch=use_gpu,
    device=device,
)

epsilon_auto = best_result.epsilon
n_clusters_auto = best_result.n_clusters
eps = epsilon_auto
n_clusters = n_clusters_auto

print(f"  Best parameters: ε = {epsilon_auto:.4f}, clusters = {n_clusters_auto} (MSE = {best_result.mse:.4f})")
K = compute_gaussian_kernel(
    X,
    epsilon=eps,
    N=X.shape[1],
    use_torch=use_torch,
    device=device,
    return_torch=False,
)
P = construct_transition_matrix(K)
evals, evecs = compute_eigendecomposition(
    P,
    n_components=min(n_components, X.shape[0] - 1),
    use_torch=use_torch,
    device=device,
    return_torch=False,
)
evals = np.real_if_close(evals)
evecs = np.real_if_close(evecs)

n_dims = min(10, len(evals) - 1)
embedding = construct_diffusion_map_embedding(
    evals, evecs, n_dimensions=n_dims, t=t, skip_first=True
)

embedding = np.real_if_close(embedding)

kmeans = KMeans(n_clusters=n_clusters, n_init=1000, random_state=0)
cluster_labels = kmeans.fit_predict(embedding)

mse_val = adjusted_mse(K, cluster_labels)
print(mse_val)   
# print(len(cluster_labels[cluster_labels == 1]))
print(evals[:40]**t)

# plot_mask = filter_diffusion_outliers(embedding[:, :2], threshold=threshold)
# if not np.any(plot_mask):
#     plot_mask = np.ones(len(embedding), dtype=bool)
# removed = np.count_nonzero(~plot_mask)
# if removed > 0:
#     print(f"Removed {removed} diffusion-map outlier(s) prior to plotting.")

plot_embedding = embedding
plot_labels = cluster_labels

sector_colors = {
    '(0,0)': 'red',
    '(1,0)': 'blue', 
    '(0,1)': 'green',
    '(-1,0)': 'purple',
    '(0,-1)': 'orange'
}

plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    plot_embedding[:, 0],
    plot_embedding[:, 1],
    c=plot_labels,
    cmap='tab10',
    s=60,
    alpha=0.75,
    edgecolors='black',
    linewidth=0.5,
)

plt.xlabel('Diffusion Coordinate φ₁', fontsize=12)
plt.ylabel('Diffusion Coordinate φ₂', fontsize=12)
plt.title(f'Diffusion Map Embedding (ε = {eps:.3f}, k = {n_clusters})', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

legend_handles = scatter.legend_elements()[0]
legend_labels = [f'Cluster {i}' for i in range(n_clusters)]
plt.legend(legend_handles, legend_labels, title='k-means labels', loc='best')

plt.tight_layout()
plt.savefig('fig3_diffusion_embedding_auto.png', dpi=150, bbox_inches='tight')
print("✓ Saved: fig3_diffusion_embedding_auto.png")
plt.show()

print("Key finding:")
print("  The automatic heuristic recovers well-separated clusters in diffusion space without manual tuning.")




/venv/main/lib/python3.12/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (5). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/venv/main/lib/python3.12/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (5). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/venv/main/lib/python3.12/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (5). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/venv/main/lib/python3.12/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (4) found smaller than n_clusters (5). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/venv/main/lib/python3.12/site-packages/sklearn/base.py:

In [2]:
# K = compute_gaussian_kernel(
#     X,
#     epsilon=eps,
#     N=X.shape[1],
#     use_torch=use_torch,
#     device=device,
#     return_torch=False,
# )
# P = construct_transition_matrix(K)
# evals, evecs = compute_eigendecomposition(
#     P,
#     n_components=min(n_components, X.shape[0] - 1),
#     use_torch=use_torch,
#     device=device,
#     return_torch=False,
# )
# evals = np.real_if_close(evals)
# evecs = np.real_if_close(evecs)

# n_dims = min(10, len(evals) - 1)
# embedding = construct_diffusion_map_embedding(
#     evals, evecs, n_dimensions=n_dims, t=t, skip_first=True
# )

# embedding = np.real_if_close(embedding)

# kmeans = KMeans(n_clusters=n_clusters, n_init=1000, random_state=0)
# cluster_labels = kmeans.fit_predict(embedding)

# mse_val = adjusted_mse(K, cluster_labels)
# print(mse_val)   
# # print(len(cluster_labels[cluster_labels == 1]))
# print(evals[:10]**t)

# plot_mask = filter_diffusion_outliers(embedding[:, :2], threshold=threshold)
# if not np.any(plot_mask):
#     plot_mask = np.ones(len(embedding), dtype=bool)
# removed = np.count_nonzero(~plot_mask)
# if removed > 0:
#     print(f"Removed {removed} diffusion-map outlier(s) prior to plotting.")

# plot_embedding = embedding[plot_mask]
# plot_labels = cluster_labels[plot_mask]

# sector_colors = {
#     '(0,0)': 'red',
#     '(1,0)': 'blue', 
#     '(0,1)': 'green',
#     '(-1,0)': 'purple',
#     '(0,-1)': 'orange'
# }

# plt.figure(figsize=(10, 8))

# scatter = plt.scatter(
#     plot_embedding[:, 0],
#     plot_embedding[:, 1],
#     c=plot_labels,
#     cmap='tab10',
#     s=60,
#     alpha=0.75,
#     edgecolors='black',
#     linewidth=0.5,
# )

# plt.xlabel('Diffusion Coordinate φ₁', fontsize=12)
# plt.ylabel('Diffusion Coordinate φ₂', fontsize=12)
# plt.title(f'Diffusion Map Embedding (ε = {eps:.3f}, k = {n_clusters})', fontsize=14, fontweight='bold')
# plt.grid(True, alpha=0.3)

# legend_handles = scatter.legend_elements()[0]
# legend_labels = [f'Cluster {i}' for i in range(n_clusters)]
# plt.legend(legend_handles, legend_labels, title='k-means labels', loc='best')

# plt.tight_layout()
# plt.savefig('fig3_diffusion_embedding_auto.png', dpi=150, bbox_inches='tight')
# print("✓ Saved: fig3_diffusion_embedding_auto.png")
# plt.show()

# print("Key finding:")
# print("  The automatic heuristic recovers well-separated clusters in diffusion space without manual tuning.")